# Семинар 12. Месяц маленькой кофейни

У кофейни возле кампуса четыре напитка: эспрессо, американо, капучино и латте.
Владелец прислал продажи за несколько недель и цены. Поможем ему разобраться,
какие дни удачные, что покупают чаще и как изменение цен повлияет на выручку.

**На занятие — 90 минут.** Работаем с одним набором данных от первого массива
до небольшого отчёта. Данные учебные, сгенерированные; цены весь период постоянны.
Числа у разных вариантов могут отличаться, способы решения остаются теми же.

| Минуты | Что делаем |
|---|---|
| 0–10 | Загрузка, массив и его форма |
| 10–25 | Индексы, срезы и копирование |
| 25–40 | Булевы маски |
| 40–55 | Двумерные массивы и оси |
| 55–65 | Цены и broadcasting |
| 65–80 | Отчёт в Pandas |
| 80–90 | Самостоятельная задача и обсуждение |

В заданиях 1–10 используйте операции NumPy без циклов по элементам.
В заданиях 11–14 используйте Pandas. Задания 16–17 — запас для тех, кто идёт быстрее.

В ячейках с заданиями заменяйте `...` своим кодом, затем запускайте проверку под ними.
До решения проверки падают — это нормально. После исправления перезапустите
ячейку с решением и её проверку. Если случайно изменили исходные данные,
перезапустите ядро и выполните ячейки заново сверху вниз.


## Перед началом

Из корня репозитория один раз установите окружение:

```bash
uv sync --extra notebook
```

Откройте этот файл в VS Code или Jupyter и выберите ядро Python из `.venv` репозитория.
Готовые CSV уже лежат в `seminar12/data`. Для другого варианта из корня выполните:

```bash
uv run python seminar12/generate_data.py --seed 17 --days 28
```

Генератор заменяет только `sales.csv` и `prices.csv`. После его запуска
перезапустите ядро ноутбука. Одинаковый seed при остальных одинаковых параметрах
создаёт одинаковые данные. Генератор работает и без NumPy/Pandas.

Следующая ячейка — готовая загрузка. Её пока достаточно выполнить.
Получим `cups` — общие продажи по дням, `sales` — продажи по напиткам,
`prices` — цены, `dates` — даты, `drinks` — названия напитков,
`weekend_mask` — отметки суббот и воскресений.


In [ ]:
import csv
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Поддерживаются запуск из корня репозитория и запуск из seminar12.
seminar_dir = Path.cwd() if Path("generate_data.py").is_file() else Path.cwd() / "seminar12"
data_dir = seminar_dir / "data"

with (data_dir / "prices.csv").open(encoding="utf-8", newline="") as stream:
    price_rows = list(csv.DictReader(stream))
with (data_dir / "sales.csv").open(encoding="utf-8", newline="") as stream:
    sale_rows = list(csv.DictReader(stream))

drinks = [row["drink"] for row in price_rows]
prices = np.array([float(row["price"]) for row in price_rows])
dates = np.array([row["date"] for row in sale_rows])
raw_sales = [[int(row[drink]) for drink in drinks] for row in sale_rows]
sales = np.array(raw_sales, dtype=np.int64)
cups = np.array([sum(row) for row in raw_sales], dtype=np.int64)
weekend_mask = np.array([date.fromisoformat(day).weekday() >= 5 for day in dates])

print("Напитки:", drinks)
print("Цены:", prices)
print("Даты:", dates[:7])
print("Всего чашек за день:", cups[:7])

## 1. Один массив — продажи по дням · 0–10 минут

`ndarray` хранит элементы одного типа. `cups.shape` — кортеж размеров,
`cups.ndim` — число осей, `cups.dtype` — тип элементов. У вектора форма `(n,)`.

Операции с массивом выполняются поэлементно. У списка `items * 2` повторяет
элементы, у массива — умножает каждое число. Сначала предскажите вывод примера.


In [ ]:
example = np.array([10, 20, 30])
print([10, 20, 30] * 2)
print(example * 2)
print(cups.shape, cups.ndim, cups.dtype)

### Задание 1. Если спрос удвоится

Получите массив `doubled`: продажи каждого дня, умноженные на два.
Исходный `cups` должен остаться прежним. Сколько элементов будет в результате?


In [ ]:
doubled = ...

In [ ]:
assert isinstance(doubled, np.ndarray), "Нужен массив NumPy"
assert doubled.shape == cups.shape
assert doubled.tolist() == [value * 2 for value in cups.tolist()]
assert cups.tolist() == [sum(row) for row in raw_sales], "Исходные продажи изменились"
print("Верно: форма сохранилась, каждое значение удвоилось")

## 2. Часть периода и общая память · 10–25 минут

Индексы начинаются с нуля. `a[-1]` — последний элемент.
Срез `a[start:stop:step]` не включает `stop`; пропущенные границы означают
начало и конец массива. `a[::-1]` разворачивает порядок.

Например, `a[1:4]` берёт элементы с номерами 1, 2, 3. Срез может быть короче
ожидаемого, если данных мало: это не ошибка.


### Задание 2. Три коротких запроса

Создайте `first_three` — продажи первых трёх дней, `last_two` — последних двух,
`every_other` — продажи через день, начиная с первого.
Используйте срезы, без перечисления отдельных индексов.


In [ ]:
first_three = ...
last_two = ...
every_other = ...

In [ ]:
assert isinstance(first_three, np.ndarray)
assert isinstance(last_two, np.ndarray)
assert isinstance(every_other, np.ndarray)
assert first_three.tolist() == cups.tolist()[:3]
assert last_two.tolist() == cups.tolist()[-2:]
assert every_other.tolist() == cups.tolist()[::2]
print("Верно: все три среза совпали")

**Сначала предскажите результат.** Срез NumPy — обычно окно (`view`) в ту же
память. Изменение через окно видно исходному массиву. `.copy()` создаёт
отдельный массив. Индексация одного элемента одномерного числового массива
возвращает отдельное скалярное значение.


In [ ]:
sample = np.array([10, 20, 30, 40])
window = sample[1:3]
separate = sample[1:3].copy()
number = sample[1]
sample[1] = 99
print("Исходный:", sample)
print("Срез:", window)
print("Копия:", separate)
print("Одно число:", number)

### Задание 3. Безопасная правка

Владелец хочет временно обнулить первый день в небольшом фрагменте данных.
Создайте `safe_fragment` — **независимую копию** первых трёх дней.
Следующая строка уже меняет её первый элемент. Исходный `cups` менять нельзя.


In [ ]:
safe_fragment = ...
safe_fragment[0] = 0

In [ ]:
assert isinstance(safe_fragment, np.ndarray)
assert not np.shares_memory(safe_fragment, cups), "Получился срез с общей памятью"
assert safe_fragment.tolist() == [0, *cups.tolist()[1:3]]
assert cups.tolist() == [sum(row) for row in raw_sales], "cups изменился: загрузите данные заново"
print("Верно: фрагмент изменился независимо")

## 3. Найдём тихие дни · 25–40 минут

Сравнение массива с числом даёт массив `bool` той же формы — булеву маску.
Маска отвечает на вопрос для каждого элемента. Индексация маской оставляет
только элементы с `True`. При суммировании маски `True` считается за 1,
`False` — за 0. Полученный скаляр можно превратить в обычный `int`.

Два условия над массивами соединяют через `&` и `|`, отрицание — `~`.
Каждое сравнение берут в скобки: `(a >= 10) & (a < 20)`.


In [ ]:
example = np.array([5, 12, 8, 20])
mask = example < 10
print(mask)
print(example[mask])
print(int(mask.sum()))

### Задание 4. Отметим тихие дни

Будем считать день тихим, если продано **строго меньше 170 чашек**.
Создайте `quiet_mask`, сохранив по одной отметке на каждый день.


In [ ]:
quiet_limit = 170
quiet_mask = ...

In [ ]:
assert isinstance(quiet_mask, np.ndarray)
assert quiet_mask.dtype == np.bool_
assert quiet_mask.shape == cups.shape
assert quiet_mask.tolist() == [value < quiet_limit for value in cups.tolist()]
print("Верно: маска готова")

### Задание 5. Выберем и посчитаем

Получите `quiet_days` — продажи только тихих дней, и `quiet_count` — их число
как обычный `int`. Что получится, если тихих дней нет?


In [ ]:
quiet_days = ...
quiet_count = ...

In [ ]:
assert isinstance(quiet_days, np.ndarray)
assert type(quiet_count) is int
assert quiet_days.tolist() == [value for value in cups.tolist() if value < quiet_limit]
assert quiet_count == len(quiet_days)
print("Тихих дней:", quiet_count, "; продажи:", quiet_days)

## 4. День × напиток · 40–55 минут

Теперь используем `sales`: каждая строка — день, каждый столбец — напиток
в порядке `drinks`. По умолчанию форма `(28, 4)`, но число дней можно менять.

`a[i, j]` — элемент, `a[i, :]` — строка, `a[:, j]` — столбец.
Двоеточие означает «все элементы этой оси». Маской строк можно выбрать
сразу несколько дней вместе со всеми напитками.


In [ ]:
print("Форма:", sales.shape)
print("Первые дни:", sales[:3])
print("Порядок столбцов:", drinks)

### Задание 6. Один напиток и выходные

Получите `espresso_sales` — продажи первого напитка за все дни,
и `weekend_sales` — строки всех суббот и воскресений. Готовая `weekend_mask`
соответствует строкам `sales`; привязываться к номеру первого выходного не нужно.
Предскажите число осей у каждого результата.


In [ ]:
espresso_sales = ...
weekend_sales = ...

In [ ]:
assert isinstance(espresso_sales, np.ndarray)
assert isinstance(weekend_sales, np.ndarray)
assert espresso_sales.shape == (len(dates),)
assert espresso_sales.tolist() == [row[0] for row in raw_sales]
assert weekend_sales.shape == (int(weekend_mask.sum()), len(drinks))
assert weekend_sales.tolist() == [
    row for row, flag in zip(raw_sales, weekend_mask, strict=False) if flag
]
print("Формы:", espresso_sales.shape, weekend_sales.shape)

### Какая ось исчезает?

Редукция сводит несколько чисел к одному. Без `axis` метод `sum()` складывает
все элементы. С `axis` он убирает указанную ось:

- для `(дни, напитки)` при `axis=0` исчезают дни: остаётся результат по напиткам;
- при `axis=1` исчезают напитки: остаётся результат по дням.

Точно так же работают `mean`, `min`, `max`. **Перед вычислением назовите форму ответа.**


In [ ]:
small = np.array([[1, 2, 3], [10, 20, 30]])
print(small.sum())
print(small.sum(axis=0))
print(small.sum(axis=1))

### Задание 7. Два итога одной таблицы

Создайте `daily_totals` — общее число чашек каждого дня,
и `drink_totals` — продажи каждого напитка за весь период.
Проверьте: сумма каждого из результатов должна дать одно и то же число чашек.


In [ ]:
daily_totals = ...
drink_totals = ...

In [ ]:
assert isinstance(daily_totals, np.ndarray)
assert isinstance(drink_totals, np.ndarray)
assert daily_totals.shape == (len(dates),)
assert drink_totals.shape == (len(drinks),)
assert daily_totals.tolist() == cups.tolist()
assert drink_totals.tolist() == [sum(column) for column in zip(*raw_sales, strict=False)]
assert daily_totals.sum() == drink_totals.sum()
print("Всего по напиткам:", dict(zip(drinks, drink_totals.tolist(), strict=False)))

`max()` возвращает **значение**, `argmax()` — **позицию** первого максимума.
По этой позиции можно достать дату или название. При равных значениях
выбирается первое. Для минимума есть `min()` и `argmin()`.


### Задание 8. Средние продажи и лучший день

Получите `drink_means` — среднее число чашек каждого напитка за день.
Найдите `best_day` — номер дня с наибольшим общим числом чашек, как обычный `int`.
При равенстве нужен первый день. Выведите его дату через `dates`.


In [ ]:
drink_means = ...
best_day = ...

In [ ]:
assert isinstance(drink_means, np.ndarray)
assert drink_means.shape == (len(drinks),)
np.testing.assert_allclose(
    drink_means, [sum(col) / len(raw_sales) for col in zip(*raw_sales, strict=False)]
)
assert type(best_day) is int
assert best_day == cups.tolist().index(max(cups.tolist()))
print("Лучший день:", dates[best_day], "; чашек:", daily_totals[best_day])

## 5. От чашек к рублям · 55–65 минут

У `sales` форма `(число дней, 4)`, у `prices` — `(4,)`.
При арифметике NumPy сравнивает формы справа налево: размеры должны совпадать
или один из них должен быть равен 1; отсутствующую ось считают единичной.
Это **broadcasting**. Вектор цен применяется к каждой строке продаж.

Например, таблица `(3, 4)` и вектор `(4,)` совместимы. Таблица `(3, 4)`
и вектор `(3,)` — нет. Формы `(3,)` и `(3, 1)` различаются!


In [ ]:
small_sales = np.array([[2, 3], [4, 1], [0, 5]])
small_prices = np.array([100.0, 200.0])
print(small_sales.shape, small_prices.shape)
print(small_sales * small_prices)

### Задание 9. Выручка каждого напитка за каждый день

Создайте `revenue` той же формы, что и `sales`: в каждой клетке — количество
чашек, умноженное на цену соответствующего напитка. Используйте одно умножение,
без циклов и без ручного повторения цен.


In [ ]:
revenue = ...

In [ ]:
assert isinstance(revenue, np.ndarray)
assert revenue.shape == sales.shape
expected = [
    [quantity * price for quantity, price in zip(row, prices.tolist(), strict=False)]
    for row in raw_sales
]
np.testing.assert_allclose(revenue, expected)
print("Выручка за первые дни:", revenue[:3])

### Задание 10. Дневная выручка

Получите `daily_revenue` — выручку каждого дня по всем напиткам.
Сначала определите, какую ось нужно убрать.


In [ ]:
daily_revenue = ...

In [ ]:
assert isinstance(daily_revenue, np.ndarray)
assert daily_revenue.shape == cups.shape
np.testing.assert_allclose(daily_revenue, [sum(row) for row in revenue.tolist()])
print("Выручка за период:", float(daily_revenue.sum()), "руб.")

## 6. Читаемый отчёт в Pandas · 65–80 минут

`DataFrame` — таблица с именами колонок и индексом строк, `Series` — одна колонка.
Создать таблицу можно из словаря: ключ — название колонки, значение — массив.
Все массивы должны иметь одинаковую длину.

```python
example = pd.DataFrame({"name": ["Аня", "Боря"], "score": [7, 9]})
example["score"]           # Series
example[["score"]]         # DataFrame из одной колонки
example.head(1)            # первая строка таблицы
```

`pd.read_csv(path)` сразу читает CSV в таблицу. Здесь начнём с массивов,
которые уже посчитали, а в конце сравним с чтением исходного файла.


### Задание 11. Соберём отчёт

Создайте `report` с тремя колонками в этом порядке:
`date` — даты из `dates`, `cups` — чашки из `daily_totals`,
`revenue` — рубли из `daily_revenue`. Одна строка — один день.
Посмотрите на первые пять строк через `.head()`.


In [ ]:
report = ...

In [ ]:
assert isinstance(report, pd.DataFrame)
assert list(report.columns) == ["date", "cups", "revenue"]
assert report.shape == (len(dates), 3)
assert report["date"].tolist() == dates.tolist()
assert report["cups"].tolist() == cups.tolist()
np.testing.assert_allclose(report["revenue"], daily_revenue)
report.head()

Фильтрация похожа на NumPy: `table[table["score"] >= 8]` оставляет строки,
подходящие под условие. Для нескольких условий также нужны `&`, `|` и скобки.
После фильтрации сохраняются исходные метки индекса.


### Задание 12. Дни с хорошим спросом

Создайте `busy`: строки `report`, в которых продано не меньше 200 чашек.
Сохраните все три колонки и исходный порядок дней. Результат может быть пустым.


In [ ]:
busy = ...

In [ ]:
assert isinstance(busy, pd.DataFrame)
assert list(busy.columns) == list(report.columns)
expected_positions = [i for i, value in enumerate(cups.tolist()) if value >= 200]
assert busy.index.tolist() == expected_positions
pd.testing.assert_frame_equal(busy, report.iloc[expected_positions])
busy.head()

`sort_values("column", ascending=False)` сортирует по убыванию.
`.head(k)` оставляет первые `k` строк. `kind="stable"` сохраняет исходный
порядок строк с одинаковым значением. Индекс после сортировки сам не перенумеровывается.


### Задание 13. Три лучших дня по выручке

Получите `top_days`: не больше трёх строк `report` с наибольшей выручкой,
от большей к меньшей. При равной выручке сохраните исходный порядок дней.
Достаточно цепочки сортировки и `.head()`.


In [ ]:
top_days = ...

In [ ]:
assert isinstance(top_days, pd.DataFrame)
expected_positions = sorted(range(len(dates)), key=lambda i: -float(daily_revenue[i]))[:3]
pd.testing.assert_frame_equal(top_days, report.iloc[expected_positions])
top_days

**Позиция и метка — разные вещи.** `.iloc[0]` берёт первую строку текущей
таблицы. `.loc[0]` ищет строку с меткой индекса 0. После сортировки это могут
быть разные дни. Предскажите результат примера до запуска.


In [ ]:
example_report = pd.DataFrame({"day": ["пн", "вт", "ср"], "revenue": [100, 300, 200]})
ordered = example_report.sort_values("revenue", ascending=False)
print(ordered)
print("Первая по позиции:", ordered.iloc[0]["day"])
print("С меткой 0:", ordered.loc[0, "day"])

Новая колонка создаётся присваиванием: `table["new"] = ...`.
Операции над числовыми колонками поэлементные. Если исходную таблицу нужно
сохранить, сначала сделайте явную `.copy()` и меняйте её.


### Задание 14. Рубли и тысячи рублей

Создайте независимую копию `report` под именем `report_in_thousands`.
Добавьте в неё `revenue_thousands` — выручку в тысячах рублей.
В исходном `report` новая колонка появляться не должна.


In [ ]:
report_in_thousands = ...
# Добавьте колонку revenue_thousands.

In [ ]:
assert isinstance(report_in_thousands, pd.DataFrame)
assert report_in_thousands is not report
assert list(report.columns) == ["date", "cups", "revenue"]
assert list(report_in_thousands.columns) == [*report.columns, "revenue_thousands"]
pd.testing.assert_frame_equal(report_in_thousands[list(report.columns)], report)
np.testing.assert_allclose(report_in_thousands["revenue_thousands"], daily_revenue / 1000)
report_in_thousands.head()

## 7. Самостоятельно: новые цены · 80–90 минут

Владелец рассматривает новый прайс. Эспрессо остаётся по прежней цене,
американо дорожает на 20 рублей, капучино на 30, латте на 40.

Предположим, что число проданных чашек не изменится. Это учебный сценарий:
в реальности изменение цены может изменить спрос.


### Задание 15. Какой день выиграл бы больше всего?

Ниже уже задан `new_prices`. Посчитайте:

1. `new_daily_revenue` — выручку каждого дня с новыми ценами;
2. `gain` — прирост относительно прежней дневной выручки;
3. `best_gain_day` — номер дня с наибольшим приростом, обычный `int`.
   При равенстве нужен первый день.

Используйте NumPy. Исходные `sales` и `prices` менять нельзя.
Выведите дату, прежнюю выручку и новую для выбранного дня.


In [ ]:
new_prices = prices + np.array([0, 20, 30, 40])
new_daily_revenue = ...
gain = ...
best_gain_day = ...

In [ ]:
assert isinstance(new_daily_revenue, np.ndarray)
assert isinstance(gain, np.ndarray)
assert new_daily_revenue.shape == gain.shape == cups.shape
expected_new = [
    sum(q * p for q, p in zip(row, new_prices.tolist(), strict=False)) for row in raw_sales
]
np.testing.assert_allclose(new_daily_revenue, expected_new)
np.testing.assert_allclose(
    gain, [new - old for new, old in zip(expected_new, daily_revenue.tolist(), strict=False)]
)
assert type(best_gain_day) is int
assert best_gain_day == gain.tolist().index(max(gain.tolist()))
assert sales.tolist() == raw_sales
assert prices.tolist() == [float(row["price"]) for row in price_rows]
print("Дата:", dates[best_gain_day], "; прирост:", float(gain[best_gain_day]), "руб.")

## Запас: если основной маршрут уже закончен

Эти задания не обязательны для завершения занятия. У каждого есть небольшая
подсказка по новому инструменту.

### Среднее по группам

`groupby` разбивает строки по значению колонки. Затем нужно указать,
что посчитать: например, `table.groupby("group")["score"].mean()`.
Сама группировка ничего не суммирует.


### Дополнительно 16. Будни против выходных

Создайте копию `report` под именем `by_day_type`, добавьте колонку
`is_weekend` из `weekend_mask`. Получите `mean_by_type` — `Series` средней
дневной выручки для будней (`False`) и выходных (`True`).
Если в данных нет дней одного из типов, для него не будет группы.
Сравните именно средние: будней и выходных разное количество.


In [ ]:
by_day_type = ...
mean_by_type = ...

In [ ]:
assert isinstance(by_day_type, pd.DataFrame)
assert by_day_type["is_weekend"].tolist() == weekend_mask.tolist()
assert isinstance(mean_by_type, pd.Series)
assert mean_by_type.index.tolist() == sorted(set(weekend_mask.tolist()))
for flag, mean_value in mean_by_type.items():
    group = [
        value
        for value, is_weekend in zip(daily_revenue.tolist(), weekend_mask.tolist(), strict=False)
        if is_weekend == flag
    ]
    assert np.isclose(mean_value, sum(group) / len(group))
assert "is_weekend" not in report.columns
mean_by_type

### Скидка сразу на всю строку

`np.where(condition, yes, no)` выбирает значение для каждого элемента.
Например, `np.where(np.array([True, False]), 0.9, 1.0)` даёт `[0.9, 1.0]`.

У дневных коэффициентов форма `(n,)`, у таблицы выручки `(n, 4)`.
Чтобы коэффициент применился к **строке**, нужна форма `(n, 1)`.
`a[:, np.newaxis]` добавляет такую ось; `None` означает то же самое.


### Дополнительно 17. Скидка в тихие дни

Предположим, что в тихие дни на все напитки действует скидка 10%, а спрос
остаётся прежним. Создайте `factors` — дневные коэффициенты 0.9 или 1.0
через `np.where`. Получите `discounted_revenue` формы `sales.shape`,
применив коэффициенты к строкам прежней `revenue`.


In [ ]:
factors = ...
discounted_revenue = ...

In [ ]:
assert isinstance(factors, np.ndarray)
assert factors.shape == cups.shape
assert isinstance(discounted_revenue, np.ndarray)
assert discounted_revenue.shape == sales.shape
expected_factors = [0.9 if value < quiet_limit else 1.0 for value in cups.tolist()]
np.testing.assert_allclose(factors, expected_factors)
np.testing.assert_allclose(
    discounted_revenue,
    [
        [value * factor for value in row]
        for row, factor in zip(revenue.tolist(), expected_factors, strict=False)
    ],
)
print("Выручка после скидок:", float(discounted_revenue.sum()), "руб.")

## Обсудим результат

- Что хранит одна строка `sales` и почему у `drink_totals` другая форма?
- Чем отличаются булева маска и выбранные по ней значения?
- Почему изменение среза может затронуть исходный массив?
- Почему `sales * prices` работает без цикла, а дневным коэффициентам нужна новая ось?
- Чем отличается `max()` от `argmax()`, а `loc` от `iloc`?
- Какие выводы меняются при другом seed, а какие свойства данных сохраняются?

Теперь можно прочитать исходные продажи напрямую в Pandas.
В этой таблице строка — день, четыре числовых столбца — отдельные напитки.
Это другие колонки, чем в нашем итоговом `report`, где уже посчитаны суммы.


In [ ]:
source_table = pd.read_csv(data_dir / "sales.csv")
source_table.head()